In [ ]:
#| default_exp misc.conv_decomposer

In [ ]:
#| include: false
from nbdev.showdoc import *

## Overview

The `Conv_Decomposer` class reduces model size and FLOPs by factorizing Conv2d layers into three smaller convolutions using Tucker decomposition. This is the Conv2d counterpart of `FC_Decomposer` (which uses SVD for Linear layers).

**How it works:** A Conv2d weight `[C_out, C_in, H, W]` is decomposed into:
1. `Conv2d(C_in, R_in, 1)` — pointwise input channel compression
2. `Conv2d(R_in, R_out, (H, W))` — spatial convolution at reduced rank
3. `Conv2d(R_out, C_out, 1)` — pointwise output channel expansion

### When to Use

| Scenario | Recommendation |
|----------|----------------|
| Large 3x3 or larger convolutions | **Highly recommended** — significant FLOP savings |
| 1x1 pointwise convolutions | Skipped automatically (already minimal) |
| Depthwise / grouped convolutions | Skipped (Tucker assumes standard convolution) |
| First layer (C_in=3) | Works but limited benefit |
| Post-training compression | Fine-tune after decomposition for best accuracy |

In [ ]:
#| export
import torch
import torch.nn as nn
import copy

In [ ]:
#| export
from fasterai.misc.fc_decomposer import _rank_from_energy, _should_decompose

def _unfold(tensor, mode):
    "Unfold a tensor along a mode into a matrix"
    return tensor.moveaxis(mode, 0).flatten(1)

def _partial_tucker(weight, ranks, n_iter=10, tol=1e-4):
    "Partial Tucker decomposition on modes [0, 1] via alternating SVD (HOOI)"
    U0 = torch.linalg.svd(_unfold(weight, 0), full_matrices=False)[0][:, :ranks[0]]
    U1 = torch.linalg.svd(_unfold(weight, 1), full_matrices=False)[0][:, :ranks[1]]

    for _ in range(n_iter):
        U0_prev, U1_prev = U0.clone(), U1.clone()
        proj = torch.einsum('oihw, or -> rihw', weight, U0)
        U1 = torch.linalg.svd(_unfold(proj, 1), full_matrices=False)[0][:, :ranks[1]]
        proj = torch.einsum('oihw, is -> oshw', weight, U1)
        U0 = torch.linalg.svd(_unfold(proj, 0), full_matrices=False)[0][:, :ranks[0]]
        if (U0 - U0_prev).norm() + (U1 - U1_prev).norm() < tol: break

    core = torch.einsum('oihw, or, is -> rshw', weight, U0, U1)
    return core, [U0, U1]

VALID_METHODS = frozenset({'tucker', 'svd'})

class Conv_Decomposer:
    "Decompose Conv2d layers to reduce parameters and FLOPs"

    def __init__(self): pass

    def decompose(self,
                  model: nn.Module,                       # The model to decompose
                  percent_removed: float = 0.5,           # Fraction of rank to remove [0, 1)
                  method: str = 'tucker',                 # 'tucker' (3 layers) or 'svd' (2 layers)
                  energy_threshold: float | None = None,  # Auto rank via energy retention (0-1)
                  layers: list[str] | None = None,        # Layer names to decompose (None = all eligible)
                  exclude: list[str] | None = None,       # Layer names to skip
                  n_iter: int = 10,                       # Max HOOI iterations (tucker only)
                  tol: float = 1e-4,                      # HOOI convergence tolerance (tucker only)
    ) -> nn.Module:
        "Decompose eligible Conv2d layers using Tucker (3 layers) or SVD (2 layers)."
        if method not in VALID_METHODS:
            raise ValueError(f"method must be one of {VALID_METHODS}, got {method!r}")
        if energy_threshold is None and not (0 <= percent_removed < 1):
            raise ValueError(f"percent_removed must be in range [0, 1), got {percent_removed}")
        if energy_threshold is not None and not (0 < energy_threshold <= 1):
            raise ValueError(f"energy_threshold must be in range (0, 1], got {energy_threshold}")

        new_model = copy.deepcopy(model)
        for name, module in list(new_model.named_modules()):
            if (isinstance(module, nn.Conv2d) and module.groups == 1 
                and min(module.kernel_size) > 1
                and _should_decompose(name, layers, exclude)):
                parent_name, _, child_name = name.rpartition('.')
                parent = new_model.get_submodule(parent_name) if parent_name else new_model
                if method == 'tucker':
                    replacement = self.Tucker(module, percent_removed, energy_threshold, n_iter, tol)
                else:
                    replacement = self.SVD(module, percent_removed, energy_threshold)
                setattr(parent, child_name, replacement)
        return new_model

    def SVD(self,
            layer: nn.Conv2d,                        # The Conv2d layer to decompose
            percent_removed: float = 0.5,            # Fraction of rank to remove
            energy_threshold: float | None = None,   # Auto rank via energy retention
    ) -> nn.Sequential:
        "SVD decomposition into 2 layers: spatial at reduced rank + pointwise expansion"
        W = layer.weight.data
        C_out, C_in = W.shape[:2]
        K = layer.kernel_size

        # Reshape to 2D: (C_out, C_in*K*K), apply SVD
        W_2d = W.reshape(C_out, -1)
        U, S, Vh = torch.linalg.svd(W_2d, full_matrices=False)

        if energy_threshold is not None:
            R = _rank_from_energy(S, energy_threshold)
        else:
            R = max(1, int((1 - percent_removed) * min(C_out, C_in)))

        # Layer 1: spatial conv at reduced rank (C_in → R)
        W1 = (torch.diag(S[:R]) @ Vh[:R]).reshape(R, C_in, *K)
        first = nn.Conv2d(C_in, R, K, stride=layer.stride,
                          padding=layer.padding, dilation=layer.dilation, bias=False)
        first.weight.data = W1

        # Layer 2: pointwise expansion (R → C_out)
        W2 = U[:, :R]
        last = nn.Conv2d(R, C_out, 1, bias=layer.bias is not None)
        last.weight.data = W2.unsqueeze(-1).unsqueeze(-1)
        if layer.bias is not None:
            last.bias.data = layer.bias.data

        return nn.Sequential(first, last)

    def Tucker(self,
               layer: nn.Conv2d,                        # The Conv2d layer to decompose
               percent_removed: float = 0.5,            # Fraction of rank to remove per mode
               energy_threshold: float | None = None,   # Auto rank via energy retention
               n_iter: int = 10,                        # Max HOOI iterations
               tol: float = 1e-4,                       # HOOI convergence tolerance
    ) -> nn.Sequential:
        "Tucker decomposition into 3 layers: pointwise compress + spatial + pointwise expand"
        W = layer.weight.data
        C_out, C_in = W.shape[:2]

        if energy_threshold is not None:
            S0 = torch.linalg.svd(_unfold(W, 0), full_matrices=False)[1]
            S1 = torch.linalg.svd(_unfold(W, 1), full_matrices=False)[1]
            R_out = _rank_from_energy(S0, energy_threshold)
            R_in = _rank_from_energy(S1, energy_threshold)
        else:
            R_out = max(1, int((1 - percent_removed) * C_out))
            R_in  = max(1, int((1 - percent_removed) * C_in))

        core, (U_out, U_in) = _partial_tucker(W, [R_out, R_in], n_iter=n_iter, tol=tol)

        first = nn.Conv2d(C_in, R_in, 1, bias=False)
        first.weight.data = U_in.t().unsqueeze(-1).unsqueeze(-1)

        middle = nn.Conv2d(R_in, R_out, layer.kernel_size, stride=layer.stride,
                           padding=layer.padding, dilation=layer.dilation, bias=False)
        middle.weight.data = core

        last = nn.Conv2d(R_out, C_out, 1, bias=layer.bias is not None)
        last.weight.data = U_out.unsqueeze(-1).unsqueeze(-1)
        if layer.bias is not None:
            last.bias.data = layer.bias.data

        return nn.Sequential(first, middle, last)

In [ ]:
show_doc(Conv_Decomposer)

In [ ]:
show_doc(Conv_Decomposer.decompose)

---

## Usage Example

```python
from fasterai.misc.conv_decomposer import Conv_Decomposer
from torchvision.models import resnet18

model = resnet18(pretrained=True)
decomposer = Conv_Decomposer()
compressed = decomposer.decompose(model, percent_removed=0.5)

# Check parameter reduction
orig = sum(p.numel() for p in model.parameters())
comp = sum(p.numel() for p in compressed.parameters())
print(f"Compression: {orig/comp:.2f}x")
```

> **Note:** Tucker decomposition uses an iterative algorithm (HOOI), so even at `percent_removed=0.0` there will be small reconstruction error. Fine-tuning after decomposition is recommended.

In [ ]:
#| hide
from fastcore.test import *

decomposer = Conv_Decomposer()

# === Tucker (3 layers, default) ===
_m = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 32, 3, padding=1))
_x = torch.randn(2, 3, 8, 8)
_m_dec = decomposer.decompose(_m, percent_removed=0.5)
test_eq(_m(_x).shape, _m_dec(_x).shape)

# Tucker structure: 3 Conv2ds (1x1, KxK, 1x1)
assert isinstance(_m_dec[0], nn.Sequential)
test_eq(len(_m_dec[0]), 3)
test_eq(_m_dec[0][0].kernel_size, (1, 1))
test_eq(_m_dec[0][1].kernel_size, (3, 3))
test_eq(_m_dec[0][2].kernel_size, (1, 1))

# percent_removed=0.0 → close reconstruction
_m2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1))
_x2 = torch.randn(2, 16, 8, 8)
test_close(_m2(_x2), decomposer.decompose(_m2, 0.0)(_x2), eps=0.01)

# 1x1 and grouped skipped
assert isinstance(decomposer.decompose(nn.Sequential(nn.Conv2d(16, 32, 1)), 0.5)[0], nn.Conv2d)
assert isinstance(decomposer.decompose(nn.Sequential(nn.Conv2d(16, 16, 3, groups=16, padding=1)), 0.5)[0], nn.Conv2d)

# Bias: only last layer gets it
_dec_bias = decomposer.Tucker(nn.Conv2d(16, 32, 3, padding=1, bias=True), 0.5)
assert _dec_bias[0].bias is None and _dec_bias[1].bias is None and _dec_bias[2].bias is not None

# Stride transfer
test_eq(decomposer.Tucker(nn.Conv2d(16, 32, 3, stride=2, padding=1), 0.5)[1].stride, (2, 2))

# Validation
with ExceptionExpected(ValueError): decomposer.decompose(nn.Sequential(nn.Conv2d(3, 16, 3)), percent_removed=1.0)
with ExceptionExpected(ValueError): decomposer.decompose(nn.Sequential(nn.Conv2d(3, 16, 3)), method='bad')

# === SVD (2 layers) ===
_m_svd = decomposer.decompose(_m, 0.5, method='svd')
test_eq(_m(_x).shape, _m_svd(_x).shape)

# SVD structure: 2 Conv2ds (KxK, 1x1)
assert isinstance(_m_svd[0], nn.Sequential)
test_eq(len(_m_svd[0]), 2)
test_eq(_m_svd[0][0].kernel_size, (3, 3))  # spatial
test_eq(_m_svd[0][1].kernel_size, (1, 1))  # pointwise expansion

# SVD bias handling
_svd_bias = decomposer.SVD(nn.Conv2d(16, 32, 3, padding=1, bias=True), 0.5)
assert _svd_bias[0].bias is None and _svd_bias[1].bias is not None

# SVD stride transfer
test_eq(decomposer.SVD(nn.Conv2d(16, 32, 3, stride=2, padding=1), 0.5)[0].stride, (2, 2))

# SVD produces valid output
_m3 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1))
_x3 = torch.randn(2, 16, 8, 8)
_m3_svd = decomposer.decompose(_m3, 0.0, method='svd')
assert torch.isfinite(_m3_svd(_x3)).all()

# SVD reconstruction is approximate (rank limited to min(C_out, C_in))
test_eq(_m3(_x3).shape, _m3_svd(_x3).shape)

# === energy_threshold + layers/exclude (both methods) ===
_m4 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1))
assert decomposer.decompose(_m4, energy_threshold=0.99)[0][0].out_channels >= \
       decomposer.decompose(_m4, 0.5)[0][0].out_channels

_m5 = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 32, 3, padding=1))
assert isinstance(decomposer.decompose(_m5, 0.5, layers=['0'])[2], nn.Conv2d)
assert isinstance(decomposer.decompose(_m5, 0.5, exclude=['2'])[2], nn.Conv2d)

In [ ]:
#| hide
#| slow
from torchvision.models import resnet18

# Decompose a real ResNet-18, verify it still works
_resnet = resnet18(weights=None)
_resnet.eval()
_x = torch.randn(2, 3, 64, 64)
_out_orig = _resnet(_x)

_dec = Conv_Decomposer()
_resnet_dec = _dec.decompose(_resnet, percent_removed=0.5)
_resnet_dec.eval()
_out_dec = _resnet_dec(_x)

# Same output shape
test_eq(_out_orig.shape, _out_dec.shape)

# Outputs are finite (no NaN/Inf)
assert torch.isfinite(_out_dec).all(), "Decomposed ResNet produced non-finite outputs"

# Parameter count reduced
_orig_params = sum(p.numel() for p in _resnet.parameters())
_dec_params = sum(p.numel() for p in _resnet_dec.parameters())
assert _dec_params < _orig_params, f"Expected fewer params: {_dec_params} >= {_orig_params}"
print(f"ResNet-18: {_orig_params:,} → {_dec_params:,} params ({_orig_params/_dec_params:.2f}x compression)")

---

## See Also

- [FC Decomposer](fc_decomposer.html) - SVD decomposition for Linear layers
- [BN Folding](bn_folding.html) - Fold BatchNorm into preceding Conv/Linear layers
- [Pruner](../prune/pruner.html) - Structured pruning that removes entire filters